# Lorenz 1D Unstable Manifold at Origin (Section 9.2.1)

Compute validated Taylor series for the parameterization P(y) of W^u_loc(0)
for the Lorenz system with σ=10, β=8/3, ρ=28.

Goal: produce ā, A^(N), and Y₀/Z₀/Z₁/Z₂ bounds for export to Lean.

In [ ]:
using Pkg
Pkg.activate(".")
using RadiiPolynomial, IntervalArithmetic, LinearAlgebra, Printf

## 1. Lorenz system parameters and fixed point

In [ ]:
# Classical parameters
σ = 10.0
ρ = 28.0
β = 8/3

# Fixed point: origin
x̃ = [0.0, 0.0, 0.0]

# Jacobian at origin
Df_origin = [-σ  σ  0.0;
             ρ  -1.0  0.0;
             0.0  0.0  -β]

println("Df(0) = ")
display(Df_origin)

# Eigenvalues
evals = eigvals(Df_origin)
println("\nEigenvalues: ", evals)

## 2. Eigenvalue and eigenvector

The positive eigenvalue is λ = (-11 + √1201)/2 ≈ 11.83.
For 1D unstable manifold: M=1, L=3.

In [ ]:
# Positive eigenvalue (unique)
λ = (-11 + sqrt(1201)) / 2
@printf("λ = %.15f\n", λ)

# Eigenvector (unit)
evecs = eigvecs(Df_origin)
# Find the column corresponding to the positive eigenvalue
idx = argmax(real.(evals))
ξ_raw = real.(evecs[:, idx])
ξ = ξ_raw / norm(ξ_raw)  # unit eigenvector

@printf("ξ = [%.15f, %.15f, %.15f]\n", ξ[1], ξ[2], ξ[3])
@printf("|ξ| = %.15f\n", norm(ξ))

# Verify: Df(0)ξ = λξ
println("\nVerification Df·ξ - λξ = ", Df_origin * ξ - λ * ξ)

## 3. Taylor recursion for approximate solution ā

From p.228: for n ≥ 2,

(nλI - Df(0)) · aₙ = -RHS(a₀,...,aₙ₋₁)

where RHS involves Cauchy products of lower-order coefficients.

In [ ]:
N = 20  # Taylor order
r = 1.0  # scaling for eigenvector (a₁ = r·ξ)

# Initialize coefficient arrays: a[l, n] for l=1,2,3 and n=0,...,N
a = zeros(3, N+1)

# Mode 0: a₀ = x̃ = (0,0,0)
a[:, 1] = x̃  # Julia is 1-indexed, so a[:,1] = a₀

# Mode 1: a₁ = r·ξ
a[:, 2] = r * ξ  # a[:,2] = a₁

# Cauchy product helper: (a_l ∗ a_m)_n = Σ_{j=0}^{n} a_{l,j} * a_{m,n-j}
function cauchy_prod(al, am, n)
    s = 0.0
    for j in 0:n
        s += al[j+1] * am[n-j+1]  # 1-indexed
    end
    return s
end

# Recursion for modes n = 2, ..., N
for n in 2:N
    # RHS from (9.16)-(9.18): nonlinear terms
    # F₁: λn·a₁ₙ - σ(a₂ₙ - a₁ₙ) = 0  → no Cauchy product
    # F₂: λn·a₂ₙ - ρa₁ₙ + a₂ₙ + (a₁∗a₃)ₙ = 0
    # F₃: λn·a₃ₙ + βa₃ₙ - (a₁∗a₂)ₙ = 0
    
    # Cauchy products (only terms j=1,...,n-1 contribute since a₀=0)
    cp13 = cauchy_prod(a[1,:], a[3,:], n)  # (a₁∗a₃)ₙ  
    cp12 = cauchy_prod(a[1,:], a[2,:], n)  # (a₁∗a₂)ₙ
    
    # The matrix (nλI - Df(0)):
    M_n = n*λ*I(3) - Df_origin
    # = [nλ+σ  -σ     0  ]
    #   [-ρ    nλ+1   0  ]
    #   [0     0      nλ+β]
    
    # RHS (the terms that don't involve aₙ)
    rhs = [0.0, -cp13, cp12]  
    # Note: at x̃=0, the Cauchy products at mode n only involve j=1..n-1
    # since a₀=0, so cp13 already excludes the a₁ₙ*a₃₀ + a₁₀*a₃ₙ = 0 terms
    # Wait - a₀ = 0 so a_{l,0} = 0 for all l. The Cauchy product
    # (a₁∗a₃)ₙ = Σ_{j=0}^n a_{1,j}*a_{3,n-j}
    # The j=0 term is a_{1,0}*a_{3,n} = 0 (since a_{1,0}=0)
    # The j=n term is a_{1,n}*a_{3,0} = 0 (since a_{3,0}=0)
    # So only j=1,...,n-1 contribute — good, these are known.
    
    # Solve for aₙ
    a[:, n+1] = M_n \ rhs  # a[:,n+1] = aₙ (1-indexed)
end

println("First few coefficients:")
for n in 0:min(5,N)
    @printf("a_%d = [%.10e, %.10e, %.10e]\n", n, a[1,n+1], a[2,n+1], a[3,n+1])
end
println("...")
@printf("a_%d = [%.10e, %.10e, %.10e]\n", N, a[1,N+1], a[2,N+1], a[3,N+1])

## 4. TODO: Radii polynomial validation

Next steps:
1. Rationalize ā to ℚ
2. Compute A^(N) = approximate inverse of DF^(N)(ā)
3. Compute Y₀, Z₀, Z₁, Z₂ bounds using interval arithmetic
4. Check radii polynomial inequality
5. Export to Lean format

In [ ]:
# Placeholder for radii polynomial validation
# This will use RadiiPolynomial.jl's interval arithmetic
println("TODO: Implement radii polynomial validation")